# Notebook 3: Batch Inference

## Overview

Run batch inference on the registered champion model from Notebook 2 using Snowflake's job-based batch API (`mv.run_batch()`) on a SPCS compute pool. Output is parquet on a stage with a `_SUCCESS` sentinel, then promoted into a queryable predictions table.

### What's Covered

| Section | Topic |
|---|---|
| 1 | Setup — imports, session, handoff constants |
| 2 | Prepare input table from the `v1_test` Dataset |
| 3 | Configure SSE-encrypted output stage |
| 4 | Submit `mv.run_batch()` on the compute pool |
| 5 | Read predictions and dedupe on `RUN_ID` |

### Handoff from Notebook 1

Reads `WAFER_YIELD_TRAINING_DATASET v1_test`. Running inference on the held-out test split gives you metrics directly comparable to the training runs in Notebook 2.

### Handoff from Notebook 2

Loads `WAFER_YIELD_DEMO.ML_MODELS.WAFER_YIELD_CHAMPION v1`. If Notebook 2 registers under a different name, update the constants cell below.

### Downstream Consumers

Predictions land in `WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS`. Dashboards, Streams, or scheduled Tasks can read this table directly.

## 📘 Section 1 — Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import time
import uuid
from datetime import datetime

from snowflake.snowpark.context import get_active_session
from snowflake.ml import dataset
from snowflake.ml.registry import Registry
from snowflake.ml.model.batch import JobSpec, OutputSpec, SaveMode

print("Imports OK")


In [ ]:
# ============================================================================
# SESSION + CONSTANTS
# ============================================================================
session = get_active_session()
session.sql("USE DATABASE WAFER_YIELD_DEMO").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
session.sql("USE WAREHOUSE WAFER_DEMO_WH").collect()

# Model handoff contract — must match Notebook 02.
MODEL_DATABASE    = "WAFER_YIELD_DEMO"
MODEL_SCHEMA      = "ML_MODELS"
MODEL_NAME        = "WAFER_YIELD_CHAMPION"     # set by Notebook 02
MODEL_VERSION     = "v1"                        # set by Notebook 02

# Input / output data (inference runs on the v1_test split -> apples-to-apples with training reports)
DATASET_FQN       = "WAFER_YIELD_DEMO.RAW_DATA.WAFER_YIELD_TRAINING_DATASET"
INFERENCE_VERSION = "v1_test"
INPUT_TABLE       = "WAFER_YIELD_DEMO.INFERENCE.WAFER_INFERENCE_INPUT"
OUTPUT_STAGE      = "WAFER_YIELD_DEMO.INFERENCE.BATCH_PREDICTIONS_STAGE"
OUTPUT_TABLE      = "WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS"

COMPUTE_POOL = "WAFER_INFERENCE_POOL"    # CPU_X64_M, recommended
# COMPUTE_POOL = "WAFER_TRAINING_POOL"   # GPU pool, only if you need GPU inference

REPLICAS    = 2
NUM_WORKERS = 2
RUN_ID      = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Session OK | WH: {session.get_current_warehouse()}")
print(f"Model:       {MODEL_DATABASE}.{MODEL_SCHEMA}.{MODEL_NAME} v{MODEL_VERSION}")
print(f"Inference on: {DATASET_FQN} {INFERENCE_VERSION}")
print(f"Compute pool: {COMPUTE_POOL}  replicas={REPLICAS} workers={NUM_WORKERS}")
print(f"Run ID: {RUN_ID}")


## 📘 Section 2 — Prepare Input Table

ML Datasets can't be read directly by batch inference jobs (permission scope) — materialize the features into a regular table first. This cell also shows the expected model signature so you can confirm the input columns line up.


In [ ]:
# ============================================================================
# LOAD ML DATASET + MATERIALIZE FOR INFERENCE
# ============================================================================
ds = dataset.load_dataset(session, DATASET_FQN, version=INFERENCE_VERSION)
features_df = ds.read.to_snowpark_dataframe()

# Drop label columns — this is inference input, not training
label_cols = ["YIELD_GOOD", "YIELD_SCORE", "DOMINANT_DEFECT_TYPE", "FIRST_TIMESTAMP"]
keep_cols = [c for c in features_df.columns if c.upper() not in [x.upper() for x in label_cols]]
inference_df = features_df.select(keep_cols)

# Materialize
session.sql("CREATE SCHEMA IF NOT EXISTS WAFER_YIELD_DEMO.INFERENCE").collect()
inference_df.write.mode("overwrite").save_as_table(INPUT_TABLE)

row_count = session.table(INPUT_TABLE).count()
print(f"Input table: {INPUT_TABLE}  ({row_count:,} rows, {len(keep_cols)} columns)")


In [ ]:
# ============================================================================
# INSPECT MODEL SIGNATURE — make sure input columns match
# ============================================================================
reg = Registry(session=session, database_name=MODEL_DATABASE, schema_name=MODEL_SCHEMA)
mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)

print(mv.show_functions())


## 📘 Section 3 — Output Stage

Batch inference writes parquet to a Snowflake stage, then drops a `_SUCCESS` sentinel when the whole job finishes. Consumers should gate reads on that file — no partial data hazard.

The stage **must** use `SNOWFLAKE_SSE` (server-side encryption). Client-side encryption is not supported.


In [ ]:
# ============================================================================
# CREATE OUTPUT STAGE (SSE-encrypted)
# ============================================================================
session.sql(f'''
    CREATE STAGE IF NOT EXISTS {OUTPUT_STAGE}
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
''').collect()

# Scope this run to its own subdirectory so reruns don't collide
output_location = f"@{OUTPUT_STAGE}/run_{RUN_ID}/"
print(f"Output location: {output_location}")


## Section 4 — Submit Batch Inference

In [ ]:
# ============================================================================
# SUBMIT BATCH INFERENCE JOB
# ============================================================================
input_df = session.table(INPUT_TABLE)

print(f"Submitting batch inference job...")
t0 = time.time()

job = mv.run_batch(
    X=input_df,
    compute_pool=COMPUTE_POOL,
    output_spec=OutputSpec(
        stage_location=output_location,
        mode=SaveMode.OVERWRITE,
    ),
    job_spec=JobSpec(
        replicas=REPLICAS,
        num_workers=NUM_WORKERS,
        # function_name="predict",  # uncomment if model has multiple functions
    ),
)

print(f"Job ID: {job.id}")
print(f"Status: {job.status}")
print(f"Waiting for completion...")

job.wait()
elapsed = time.time() - t0
print(f"\nFinal status: {job.status}  |  {elapsed/60:.1f} min")


## 📘 Section 5 — Read Predictions

Gate on the `_SUCCESS` sentinel file, then read the parquet output. Output contains all original input columns **plus** the model's prediction column (`output_feature_0` for a single-output DNN).


In [ ]:
# ============================================================================
# VERIFY JOB OUTPUT + READ PARQUET
# ============================================================================
files = session.sql(f"LS {output_location}").collect()
print(f"Output files: {len(files)}")
for r in files[:10]:
    print(f"   {r['name']}  size={r['size']:,}")

success = any("_SUCCESS" in r['name'] for r in files)
has_parquet = any(r['name'].endswith('.parquet') for r in files)

print(f"\n_SUCCESS sentinel present: {success}")
print(f"Parquet files present: {has_parquet}")

assert has_parquet, "No parquet output found — check job logs with job.get_logs()"
if not success:
    print("WARNING: _SUCCESS sentinel missing — data may be in _temporary/. Proceeding with available parquet files.")

In [ ]:
# ============================================================================
# LOAD PREDICTIONS INTO A TABLE (dedupe-safe on RUN_ID)
# ============================================================================
from snowflake.snowpark.functions import lit, current_timestamp

results_df = (
    session.read
    .option("pattern", ".*[.]parquet")
    .parquet(output_location)
    .with_column("RUN_ID", lit(RUN_ID))
    .with_column("INFERENCE_TIMESTAMP_UTC", current_timestamp())
    .with_column("MODEL_NAME", lit(MODEL_NAME))
    .with_column("MODEL_VERSION", lit(MODEL_VERSION))
)

try:
    session.sql(f"DELETE FROM {OUTPUT_TABLE} WHERE RUN_ID = '{RUN_ID}'").collect()
except Exception:
    pass

results_df.write.mode("append").save_as_table(OUTPUT_TABLE)

total = session.table(OUTPUT_TABLE).filter(f"RUN_ID = '{RUN_ID}'").count()
print(f"Predictions saved to {OUTPUT_TABLE}  |  this run: {total:,} rows")

In [ ]:
# ============================================================================
# ANALYZE PREDICTIONS
# ============================================================================
preds = session.table(OUTPUT_TABLE).filter(f"RUN_ID = '{RUN_ID}'")

print("Sample predictions:")
preds.select('"WAFER_ID"', '"output_feature_0"', "MODEL_NAME", "MODEL_VERSION").show(10)

print("\nPrediction distribution:")
preds.select('"output_feature_0"').describe().show()

## Summary

| Step | What Happened |
|---|---|
| **Input** | Materialized `v1_test` dataset into an inference input table |
| **Inference** | Ran `mv.run_batch()` on SPCS compute pool with parquet + `_SUCCESS` output |
| **Output** | Predictions saved to `WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS` |

### Scheduling Options

| Tool | Pattern |
|---|---|
| **Snowflake Tasks** | Wrap this logic in a stored procedure, schedule via `CREATE TASK ... SCHEDULE = 'USING CRON ...'` |
| **Streams + Tasks** | Trigger on new rows in a wafer intake table — score continuously |